<a href="https://colab.research.google.com/github/fabhari/Algorithm-examples/blob/main/railpoint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import numpy as np

def calculate_distance(box1, box2):
    c1 = np.array([(box1[0] + box1[2]) / 2, (box1[1] + box1[3]) / 2])
    c2 = np.array([(box2[0] + box2[2]) / 2, (box2[1] + box2[3]) / 2])
    return np.linalg.norm(c1 - c2)

def detect_behaviors(detections, prev_positions, x_thresh, move_thresh, fall_ar_thresh=2.0, fall_height_ratio=0.5):
    persons = [d for d in detections if d['label'] == 'person']
    luggage = [d for d in detections if d['label'] in ['suitcase', 'backpack', 'bag']]
    phones = [d for d in detections if d['label'] == 'cell phone']
    results = []

    for person in persons:
        pid = person['id']
        p_box = person['bbox']
        behavior = {"person_id": pid, "actions": []}

        for item in luggage:
            if calculate_distance(p_box, item['bbox']) < x_thresh:
                behavior["actions"].append("carrying item")
                break

        is_using_phone = any(calculate_distance(p_box, ph['bbox']) < 40 for ph in phones)
        if pid in prev_positions:
            dist_moved = calculate_distance(p_box, prev_positions[pid])
            if is_using_phone and dist_moved > move_thresh:
                behavior["actions"].append("walking while using phone")

            # Fall logic
            curr_w, curr_h = p_box[2]-p_box[0], p_box[3]-p_box[1]
            prev_box = prev_positions[pid]
            prev_w, prev_h = prev_box[2]-prev_box[0], prev_box[3]-prev_box[1]
            if prev_h > 0 and curr_h > 0:
                if ((curr_w/curr_h) / (prev_w/prev_h) > fall_ar_thresh) and (curr_h < prev_h * fall_height_ratio):
                    behavior["actions"].append("falling down")

        results.append(behavior)
    return results

In [7]:
!pip install ultralytics opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.7 MB/s eta 0:00:00


In [8]:
# Install necessary libraries
!pip install ultralytics opencv-python-headless

In [9]:
# Install necessary libraries
!pip install ultralytics opencv-python-headless

In [10]:
import torch
from ultralytics import YOLO
import cv2

# Check if GPU is available and set device
global model, device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Load a well-trained CNN model (YOLOv8 large is more accurate but slower)
model = YOLO('yolov8l.pt')

# Move model to GPU
model.to(device)
print('YOLOv8l model loaded globally.')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using device: cuda
YOLOv8l model loaded globally.


In [16]:
!pip install ultralytics opencv-python-headless

In [17]:
import torch
from ultralytics import YOLO
import cv2

# Set device globally
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Load model globally
model = YOLO('yolov8l.pt')
model.to(device)
print('YOLOv8l model loaded successfully.')

Using device: cuda
YOLOv8l model loaded successfully.


In [14]:
# This cell allows you to tune the sensitivity of the detections
# Adjust these values if the system is too sensitive or not sensitive enough

DETECTION_SETTINGS = {
    'luggage_proximity_pixels': 100,    # Distance between person and bag
    'phone_proximity_pixels': 40,      # Distance between person and phone
    'movement_threshold': 5,           # Pixels moved to be considered 'walking'
    'fall_aspect_ratio_change': 2.0,   # Width/Height ratio increase for falls
    'fall_height_reduction': 0.5       # Height must drop to 50% for falls
}

print("Detection thresholds confirmed. These are used in the main analysis script.")

Detection thresholds confirmed. These are used in the main analysis script.


### Video Analysis Script

This cell combines the event detection and video annotation processes into a single, easy-to-use script.

**Instructions:**
1. **Specify your input video path** in the `input_video_path` variable below.
2. Run this cell.

It will output a summary of detected events (like 'person carrying a bag', 'person walking', 'person falling') and create an annotated video file in your Colab environment.

In [35]:
import os
import pandas as pd
import cv2

# Updated to use the new uploaded file
video_input = '/content/sample_data/test2.mp4'

def run_complete_analysis(video_path, settings):
    if not os.path.exists(video_path):
        print(f"[!] File not found: {video_path}")
        return

    file_base = os.path.basename(video_path)
    file_name, file_ext = os.path.splitext(file_base)
    output_file = f"annotated_{file_name}{file_ext}"

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[!] Error: Could not open {video_path}. Verify it is a valid video file.")
        return

    w, h = int(cap.get(3)), int(cap.get(4))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    out = cv2.VideoWriter(output_file, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

    prev_positions, events_log, frame_count = {}, [], 0
    print(f"Starting analysis on {file_base}...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        # Use the globally loaded model
        results = model.track(frame, persist=True, device=device, verbose=False)
        current_detections = []

        if results and results[0].boxes.id is not None:
            boxes = results[0].boxes
            for i in range(len(boxes.id)):
                det = {
                    'id': int(boxes.id[i]),
                    'label': model.names[int(boxes.cls[i])],
                    'bbox': boxes.xyxy[i].tolist()
                }
                current_detections.append(det)
                # Draw basic tracking box
                x1, y1, x2, y2 = map(int, det['bbox'])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Analyze behaviors
        behaviors = detect_behaviors(
            current_detections,
            prev_positions,
            x_thresh=settings.get('luggage_proximity_pixels', 100),
            move_thresh=settings.get('movement_threshold', 5),
            fall_ar_thresh=settings.get('fall_aspect_ratio_change', 2.0),
            fall_height_ratio=settings.get('fall_height_reduction', 0.5)
        )

        for b in behaviors:
            if b['actions']:
                for action in b['actions']:
                    events_log.append({'time': f"{frame_count/fps:.2f}s", 'person_id': b['person_id'], 'event': action})
                    # Highlight the person in red if an event is detected
                    for d in current_detections:
                        if d['id'] == b['person_id']:
                            px1, py1, px2, py2 = map(int, d['bbox'])
                            cv2.rectangle(frame, (px1, py1), (px2, py2), (0, 0, 255), 3)
                            cv2.putText(frame, f"EVENT: {action}", (px1, py2+20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        prev_positions = {d['id']: d['bbox'] for d in current_detections if d['label'] == 'person'}
        out.write(frame)
        frame_count += 1

    cap.release()
    out.release()
    print(f"\nAnalysis finished! Saved as: {output_file}")

    if events_log:
        print("\nDetected Event Summary:")
        display(pd.DataFrame(events_log).drop_duplicates())
    else:
        print("\nNo specific events were detected.")

# Run the updated process
run_complete_analysis(video_input, DETECTION_SETTINGS)

Starting analysis on test2.mp4...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 232ms
Prepared 1 package in 65ms
Installed 1 package in 6ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


Analysis finished! Saved as: annotated_test2.mp4

Detected Event Summary:


,time,person_id,event
0,0.24s,5,carrying item
1,0.24s,14,carrying item
2,0.28s,5,carrying item
3,0.28s,14,carrying item
4,0.32s,5,carrying item
...,...,...,...
165,17.12s,5,carrying item
166,17.16s,121,carrying item
167,17.20s,121,carrying item
168,18.20s,5,carrying item


In [19]:
def process_video_for_events(video_path, x_thresh=100, move_thresh=5, fall_ar_thresh=2.0, fall_height_ratio=0.5):
    import cv2
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): return []
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frame_count, prev_positions, events_log = 0, {}, []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        results = model.track(frame, persist=True, device=device, verbose=False)
        current_detections = []
        if results and results[0].boxes.id is not None:
            boxes = results[0].boxes
            for i in range(len(boxes.id)):
                current_detections.append({'id': int(boxes.id[i]), 'label': model.names[int(boxes.cls[i])], 'bbox': boxes.xyxy[i].tolist()})

        behaviors = detect_behaviors(current_detections, prev_positions, x_thresh, move_thresh, fall_ar_thresh, fall_height_ratio)
        for b in behaviors:
            for action in b['actions']:
                events_log.append({'time_offset': f"{frame_count/fps:.2f}s", 'person_id': b['person_id'], 'event': action})

        prev_positions = {d['id']: d['bbox'] for d in current_detections if d['label'] == 'person'}
        frame_count += 1
    cap.release()
    return events_log

### Note:
This cell previously contained a simulated test function and a duplicate `detect_behaviors`.
It has been removed as the `detect_behaviors` in cell `wErmQx880br2` has been enhanced, and the `process_video_for_events` in cell `deb586d5` now uses real YOLOv8 detections.

### Note:
This cell previously contained a call to an outdated test function.
It has been removed as the video processing logic in cell `deb586d5` now directly uses YOLOv8 for detection and tracking.

In [24]:
def process_video_with_annotations(video_path, output_path, x_thresh=100, move_thresh=5, fall_ar_thresh=2.0, fall_height_ratio=0.5):
    import cv2
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Error: Could not open video.")
        return

    w, h = int(cap.get(3)), int(cap.get(4))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    prev_positions = {}

    while cap.isOpened():
      ret, frame = cap.read()
      if not ret: break

      results = model.track(frame, persist=True, device=device, verbose=False)
      current_detections = []

      if results and results[0].boxes.id is not None:
          boxes = results[0].boxes
          for i in range(len(boxes.id)):
              det = {
                  'id': int(boxes.id[i]),
                  'label': model.names[int(boxes.cls[i])],
                  'bbox': boxes.xyxy[i].tolist()
              }
              current_detections.append(det)

              # Draw basic bounding box
              x1, y1, x2, y2 = map(int, det['bbox'])
              color = (0, 255, 0) # Green for general
              cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
              cv2.putText(frame, f"{det['label']} {det['id']}", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

      # Detect behaviors for annotation overlay
      behaviors = detect_behaviors(current_detections, prev_positions, x_thresh, move_thresh, fall_ar_thresh, fall_height_ratio)

      for b in behaviors:
          if b['actions']:
              # Find the person's bbox to draw the label
              for d in current_detections:
                  if d['id'] == b['person_id']:
                      px1, py1, px2, py2 = map(int, d['bbox'])
                      label_text = ", ".join(b['actions'])
                      cv2.putText(frame, f"EVENT: {label_text}", (px1, py2 + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                      cv2.rectangle(frame, (px1, py1), (px2, py2), (0, 0, 255), 3) # Highlight in Red

      prev_positions = {d['id']: d['bbox'] for d in current_detections if d['label'] == 'person'}
      out.write(frame)

    cap.release()
    out.release()